## 1. Setup & Mount Google Drive

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies
!pip install transformers scikit-learn

In [3]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

GPU Available: True
GPU Name: Tesla T4
GPU Memory: 15.83 GB


## 2. Upload Training Data to Google Drive

**Before running this notebook:**
1. Create folder in Google Drive: `ai-legal-summarizer-training/`
2. Upload these files from your local machine:
   - `train.json` (from `e:\ai-legal-summarizer\data\training_data\document_structure_training\`)
   - `dev.json` (same location)
   - `test.json` (same location)

**Update the path below to match your Drive folder:**

In [4]:
# Set your Google Drive paths
DATA_DIR = '/content/drive/MyDrive/ai-legal-summarizer-training'
OUTPUT_DIR = '/content/drive/MyDrive/ai-legal-summarizer-training/trained_model'

# Verify files exist
import os
print("Checking training files...")
for file in ['train.json', 'dev.json', 'test.json']:
    path = f"{DATA_DIR}/{file}"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"✅ {file}: {size_mb:.2f} MB")
    else:
        print(f"❌ {file}: NOT FOUND")
        print(f"   Please upload to: {DATA_DIR}")

Checking training files...
✅ train.json: 5.20 MB
✅ dev.json: 1.11 MB
✅ test.json: 1.11 MB


## 3. Training Configuration & Model

In [6]:
import json
import torch
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import Dict, List

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EvalPrediction,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from torch.utils.data import Dataset

# Training configuration
MODEL_NAME = 'bert-base-uncased'
MAX_LENGTH = 256
BATCH_SIZE = 32  # Larger batch size for GPU
LEARNING_RATE = 2e-5
NUM_EPOCHS = 5
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01

# Label mapping
LABEL_MAP = {
    'FACTS': 0,
    'ISSUES': 1,
    'LEGAL_ANALYSIS': 2,
    'REASONING': 3,
    'JUDGMENT': 4,
    'ORDERS': 5,
}

ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
NUM_LABELS = len(LABEL_MAP)

print(f"Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

Configuration:
  Model: bert-base-uncased
  Batch Size: 32
  Epochs: 5
  Device: GPU


In [8]:
# Dataset class
class DocumentStructureDataset(Dataset):
    """PyTorch Dataset for document structure classification"""

    def __init__(self, data: List[Dict], tokenizer, max_length: int = MAX_LENGTH):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        label = item['label_id']

        # Tokenize
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long),
        }

## 4. Load Training Data

In [9]:
# Load data
print("Loading training data...")

with open(f"{DATA_DIR}/train.json", 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open(f"{DATA_DIR}/dev.json", 'r', encoding='utf-8') as f:
    dev_data = json.load(f)

with open(f"{DATA_DIR}/test.json", 'r', encoding='utf-8') as f:
    test_data = json.load(f)

print(f"✅ Train: {len(train_data)} examples")
print(f"✅ Dev: {len(dev_data)} examples")
print(f"✅ Test: {len(test_data)} examples")
print(f"\nTotal: {len(train_data) + len(dev_data) + len(test_data)} examples")

Loading training data...
✅ Train: 2258 examples
✅ Dev: 483 examples
✅ Test: 485 examples

Total: 3226 examples


## 5. Initialize Model & Tokenizer

In [10]:
# Initialize tokenizer and model
print(f"Loading {MODEL_NAME}...")

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"✅ Model loaded on: {device}")

# Create datasets
train_dataset = DocumentStructureDataset(train_data, tokenizer)
dev_dataset = DocumentStructureDataset(dev_data, tokenizer)
test_dataset = DocumentStructureDataset(test_data, tokenizer)

print("✅ Datasets created")

Loading bert-base-uncased...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded on: cuda
✅ Datasets created


## 6. Define Training Function

In [11]:
def compute_metrics(pred: EvalPrediction) -> Dict:
    """Compute evaluation metrics"""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

## 7. Train Model (5-10 minutes on GPU)

In [12]:
# Training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

print("="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Training on: {device}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Estimated time: 5-10 minutes on GPU")
print("="*70)

# Train
train_result = trainer.train()

print("\n✅ Training complete!")

STARTING TRAINING
Training on: cuda
Epochs: 5
Batch size: 32
Estimated time: 5-10 minutes on GPU


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.689300,1.502429,0.285714,0.306321,0.285714,0.184008
2,1.543300,1.323874,0.432712,0.389425,0.432712,0.368497
3,1.339300,1.222495,0.490683,0.485758,0.490683,0.420023
4,1.246500,1.145788,0.583851,0.534712,0.583851,0.554785
5,1.108500,1.138886,0.583851,0.534784,0.583851,0.557692



✅ Training complete!


## 8. Evaluate on Test Set

In [13]:
# Evaluate on test set
print("="*70)
print("EVALUATING ON TEST SET")
print("="*70)

test_results = trainer.evaluate(test_dataset)

# Get predictions for detailed metrics
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)
labels = predictions.label_ids

# Classification report
print("\nDetailed Classification Report:")
print("-"*70)
report = classification_report(
    labels,
    preds,
    target_names=[ID_TO_LABEL[i] for i in range(NUM_LABELS)],
    digits=4,
)
print(report)

# Print summary
print("\n" + "="*70)
print("TRAINING SUMMARY")
print("="*70)
print(f"Training Time: {train_result.metrics['train_runtime']:.2f}s")
print(f"\nTest Set Performance:")
print(f"  Accuracy:  {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.2f}%)")
print(f"  Precision: {test_results['eval_precision']:.4f}")
print(f"  Recall:    {test_results['eval_recall']:.4f}")
print(f"  F1 Score:  {test_results['eval_f1']:.4f}")
print("="*70)

EVALUATING ON TEST SET



Detailed Classification Report:
----------------------------------------------------------------------
                precision    recall  f1-score   support

         FACTS     0.4189    0.4769    0.4460       130
        ISSUES     0.0000    0.0000    0.0000        20
LEGAL_ANALYSIS     0.6264    0.7078    0.6646       154
     REASONING     0.5460    0.6054    0.5742       147
      JUDGMENT     0.0000    0.0000    0.0000        15
        ORDERS     0.0000    0.0000    0.0000        19

      accuracy                         0.5361       485
     macro avg     0.2652    0.2984    0.2808       485
  weighted avg     0.4767    0.5361    0.5046       485


TRAINING SUMMARY
Training Time: 713.91s

Test Set Performance:
  Accuracy:  0.5361 (53.61%)
  Precision: 0.4767
  Recall:    0.5361
  F1 Score:  0.5046


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 8.5. Analyze Results (Optional)

**Understanding the results:**
- ✅ **Good sections**: LEGAL_ANALYSIS (66%), REASONING (57%), FACTS (45%)
- ❌ **Bad sections**: ISSUES, JUDGMENT, ORDERS (0% - model didn't predict these)

**Why?** These sections are rare in the training data:
- ISSUES: 20 examples (4%)
- JUDGMENT: 15 examples (3%)
- ORDERS: 19 examples (4%)

**Recommendation:** Use hybrid approach - BERT for common sections, rules for rare ones.

In [14]:
# Analyze prediction distribution
import pandas as pd
from collections import Counter

# Count predictions per section
pred_counts = Counter([ID_TO_LABEL[p] for p in preds])
label_counts = Counter([ID_TO_LABEL[l] for l in labels])

print("="*70)
print("PREDICTION ANALYSIS")
print("="*70)
print("\nTrue Labels (Test Set):")
for section, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    pct = count / len(labels) * 100
    print(f"  {section:20} : {count:3} examples ({pct:5.1f}%)")

print("\nModel Predictions:")
for section, count in sorted(pred_counts.items(), key=lambda x: -x[1]):
    pct = count / len(preds) * 100
    print(f"  {section:20} : {count:3} predictions ({pct:5.1f}%)")

print("\n" + "="*70)
print("RECOMMENDATION:")
print("="*70)
print("✅ Use BERT for: LEGAL_ANALYSIS, REASONING, FACTS (good performance)")
print("❌ Use Rules for: ISSUES, JUDGMENT, ORDERS (model didn't learn these)")
print("="*70)

PREDICTION ANALYSIS

True Labels (Test Set):
  LEGAL_ANALYSIS       : 154 examples ( 31.8%)
  REASONING            : 147 examples ( 30.3%)
  FACTS                : 130 examples ( 26.8%)
  ISSUES               :  20 examples (  4.1%)
  ORDERS               :  19 examples (  3.9%)
  JUDGMENT             :  15 examples (  3.1%)

Model Predictions:
  LEGAL_ANALYSIS       : 174 predictions ( 35.9%)
  REASONING            : 163 predictions ( 33.6%)
  FACTS                : 148 predictions ( 30.5%)

RECOMMENDATION:
✅ Use BERT for: LEGAL_ANALYSIS, REASONING, FACTS (good performance)
❌ Use Rules for: ISSUES, JUDGMENT, ORDERS (model didn't learn these)


## 9. Save Model & Results

In [15]:
# Save final model
model_path = f"{OUTPUT_DIR}/final_model"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

print(f"✅ Model saved to: {model_path}")

# Save results
results = {
    'training_args': {
        'model_name': MODEL_NAME,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'num_epochs': NUM_EPOCHS,
        'device': str(device),
    },
    'training_results': {
        'train_loss': float(train_result.training_loss),
        'train_runtime': train_result.metrics['train_runtime'],
    },
    'test_results': {
        'accuracy': float(test_results['eval_accuracy']),
        'precision': float(test_results['eval_precision']),
        'recall': float(test_results['eval_recall']),
        'f1': float(test_results['eval_f1']),
    },
    'classification_report': report,
    'timestamp': datetime.now().isoformat(),
}

results_file = f"{OUTPUT_DIR}/training_results.json"
with open(results_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved to: {results_file}")

✅ Model saved to: /content/drive/MyDrive/ai-legal-summarizer-training/trained_model/final_model
✅ Results saved to: /content/drive/MyDrive/ai-legal-summarizer-training/trained_model/training_results.json


## 10. Download Model Files

**After training completes:**
1. Go to Files tab in Colab (left sidebar)
2. Navigate to: `/content/drive/MyDrive/ai-legal-summarizer-training/trained_model/final_model/`
3. Download these files:
   - `config.json`
   - `model.safetensors` (or `pytorch_model.bin`)
   - `vocab.txt`
   - `tokenizer_config.json`
   - `special_tokens_map.json`

**Or download the entire folder as ZIP:**

In [16]:
# Create ZIP file for easy download
import shutil

zip_path = f"{OUTPUT_DIR}/document_classifier_trained"
shutil.make_archive(zip_path, 'zip', f"{OUTPUT_DIR}/final_model")

print(f"✅ Model packaged as ZIP")
print(f"📦 Download from: {zip_path}.zip")
print(f"\n🎉 Training complete! Download the model and use it in your project.")

✅ Model packaged as ZIP
📦 Download from: /content/drive/MyDrive/ai-legal-summarizer-training/trained_model/document_classifier_trained.zip

🎉 Training complete! Download the model and use it in your project.


---

## Next Steps:

1. **Download the trained model** (ZIP file or individual files)
2. **Extract to your local project**: `e:\ai-legal-summarizer\backend\models\document_classifier\final_model\`
3. **Use the model** in your application (see usage instructions in next cell)

---